# DCDI: Differentiable Causal Discovery with Instrumental Variables

## Variables for Causal Analysis

### Exogenous Variables (Features)
- `batch_rank_capped`: Courier workload (capped)
- `speed_mean`: Average speed during delivery window
- `gps_gap_min`: GPS freshness (time since last update)
- `spatial_congestion_norm`: Road congestion around delivery area
- `WSI`: Weather Severity Index
- `typecode_grouped`: Delivery type/category

### Target Variable
- `ETA_mins`: Estimated Time of Arrival (in minutes)

## Algorithm Overview
DCDI learns a Directed Acyclic Graph (DAG) that represents causal relationships. Key aspects:
- Uses PyTorch + GPU for scalability
- L1 regularization encourages sparsity in the causal graph
- DAG constraint prevents cycles
- Estimates both the causal structure and causal mechanisms

In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'adjacency_matrix': B_learned,
    'variable_names': variable_names,
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
}, 'dcdi_model_checkpoint.pt')

print("Model checkpoint saved: dcdi_model_checkpoint.pt")

# Create a summary statistics table
summary_stats = pd.DataFrame({
    'Metric': [
        'Number of Variables',
        'Number of Samples',
        'Training Epochs',
        'Final MSE Loss',
        'Graph Sparsity (%)',
        'Mean RMSE',
        'Mean R²',
        'Device Used'
    ],
    'Value': [
        n_vars,
        len(df),
        150,
        f"{loss_history['mse'][-1]:.6f}",
        f"{(1 - (len([1 for i in range(n_vars) for j in range(n_vars) if i != j and abs(B_learned[i,j]) > 0.05]) / (n_vars * (n_vars-1)))) * 100:.1f}",
        f"{errors_df['RMSE'].mean():.4f}",
        f"{errors_df['R²'].mean():.4f}",
        str(device)
    ]
})

print("\n" + "=" * 80)
print("EXPERIMENT SUMMARY")
print("=" * 80)
print(summary_stats.to_string(index=False))

In [ ]:
# Summary and interpretability report
print("\n" + "=" * 80)
print("DCDI CAUSAL DISCOVERY SUMMARY REPORT")
print("=" * 80)

report = f"""
ANALYSIS CONFIGURATION:
  • Target Variable: ETA_mins (Estimated Time of Arrival)
  • Number of Variables: {n_vars}
  • Number of Samples: {len(df)}
  • Device: {device}

TRAINING RESULTS:
  • Final MSE Loss: {loss_history['mse'][-1]:.4f}
  • Final L1 (Sparsity): {loss_history['l1'][-1]:.4f}
  • Final DAG Constraint: {loss_history['dag'][-1]:.4f}

DISCOVERED CAUSAL STRUCTURE:
  • Total Number of Edges: {len([1 for i in range(n_vars) for j in range(n_vars) 
                                  if i != j and abs(B_learned[i,j]) > 0.05])}
  • Sparsity: {(1 - (len([1 for i in range(n_vars) for j in range(n_vars) 
                           if i != j and abs(B_learned[i,j]) > 0.05]) / (n_vars * (n_vars-1)))) * 100:.1f}%

KEY FINDINGS:
  1. Direct causes of ETA_mins: {len(causal_parents)} variables with significant effects
  2. Strongest predictor: {causal_parents[0][0] if causal_parents else 'None'} 
     (effect = {causal_parents[0][1]:.4f})
  
IMPLICATIONS FOR DELIVERY OPTIMIZATION:
  • Variables to focus on for ETA improvements: Those with strongest causal effects
  • Confounding structures: Check for colliders that may affect causal inference
  • Intervention recommendations: Target variables with positive effects on ETA

MODEL QUALITY:
  • Mean RMSE across variables: {errors_df['RMSE'].mean():.4f}
  • Mean R² across variables: {errors_df['R²'].mean():.4f}
  • Best-fit variable: {errors_df.loc[errors_df['R²'].idxmax(), 'Variable']}
    (R² = {errors_df['R²'].max():.4f})
"""

print(report)

# Save report to file
with open('dcdi_causal_discovery_report.txt', 'w') as f:
    f.write(report)
    f.write("\n\n" + "=" * 80 + "\n")
    f.write("DETAILED CAUSAL EDGES\n")
    f.write("=" * 80 + "\n")
    if len(edges) > 0:
        f.write(edges_df.to_string(index=False))
    else:
        f.write("No significant edges detected\n")

print("\nReport saved to: dcdi_causal_discovery_report.txt")

In [ ]:
# Causal intervention analysis - simulate interventions on causal variables
print("\n" + "=" * 80)
print("CAUSAL INTERVENTION ANALYSIS")
print("=" * 80)
print("\nSimulating manipulative interventions on key variables...")

# Get the index of the target variable
eta_idx = variable_names.index('ETA_mins')

# Find variables with direct causal effect on ETA
causal_parents = []
for j in range(n_vars):
    if j != eta_idx and abs(B_learned[eta_idx, j]) > 0.05:
        causal_parents.append((variable_names[j], B_learned[eta_idx, j]))

causal_parents.sort(key=lambda x: abs(x[1]), reverse=True)

print(f"\nVariables with direct causal effect on ETA_mins:")
intervention_results = []

for parent_var, effect_weight in causal_parents:
    parent_idx = variable_names.index(parent_var)
    
    # Baseline prediction (mean values)
    X_baseline = X_tensor.clone()
    with torch.no_grad():
        eta_baseline = model(X_baseline)[:, eta_idx]
    
    # Intervention: increase the parent variable by 1 standard deviation
    X_intervention = X_tensor.clone()
    X_intervention[:, parent_idx] += 1.0  # +1 std
    
    with torch.no_grad():
        eta_intervention = model(X_intervention)[:, eta_idx]
    
    avg_effect = (eta_intervention - eta_baseline).mean().item()
    
    intervention_results.append({
        'Variable': parent_var,
        'Direct Effect': effect_weight,
        'Avg Change in ETA': avg_effect,
        'Interpretation': f"+1σ in {parent_var} → {avg_effect:.3f}σ change in ETA"
    })
    
    print(f"\n• {parent_var}:")
    print(f"  Direct effect weight: {effect_weight:.4f}")
    print(f"  Causal impact: +1σ change → {avg_effect:.4f}σ in ETA_mins")

if not intervention_results:
    print("\nNo direct causal effects detected on ETA_mins")
else:
    intervention_df = pd.DataFrame(intervention_results)
    print("\n" + "=" * 80)
    print("Intervention Summary:")
    print(intervention_df.to_string(index=False))

In [ ]:
# Visualize prediction quality for key variables
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

key_vars = ['ETA_mins', 'batch_rank_capped', 'speed_mean', 'WSI', 'spatial_congestion_norm', 'gps_gap_min']

for idx, var_name in enumerate(key_vars):
    if var_name in variable_names:
        var_idx = variable_names.index(var_name)
        
        ax = axes[idx]
        
        # Plot actual vs predicted
        ax.scatter(X_np[:200, var_idx], X_pred_np[:200, var_idx], alpha=0.6, s=30)
        
        # Add perfect prediction line
        min_val = min(X_np[:, var_idx].min(), X_pred_np[:, var_idx].min())
        max_val = max(X_np[:, var_idx].max(), X_pred_np[:, var_idx].max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction', linewidth=2)
        
        r2 = errors_df[errors_df['Variable'] == var_name]['R²'].values[0]
        
        ax.set_xlabel('Actual (Standardized)', fontsize=10)
        ax.set_ylabel('Predicted (Standardized)', fontsize=10)
        ax.set_title(f'{var_name}\n(R² = {r2:.3f})', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend()

plt.tight_layout()
plt.savefig('dcdi_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Prediction visualization completed")

In [ ]:
# Evaluate model fit on the data
print("\n" + "=" * 80)
print("MODEL EVALUATION")
print("=" * 80)

with torch.no_grad():
    X_pred_full = model(X_tensor)
    mse_final = nn.MSELoss()(X_tensor, X_pred_full).item()
    rmse_final = np.sqrt(mse_final)

print(f"\nFinal MSE (on all data): {mse_final:.6f}")
print(f"Final RMSE (on all data): {rmse_final:.6f}")

# Per-variable prediction errors
X_pred_np = X_pred_full.detach().cpu().numpy()
X_np = X_tensor.detach().cpu().numpy()

per_var_errors = []
for i, var_name in enumerate(variable_names):
    mse_i = np.mean((X_np[:, i] - X_pred_np[:, i]) ** 2)
    rmse_i = np.sqrt(mse_i)
    mae_i = np.mean(np.abs(X_np[:, i] - X_pred_np[:, i]))
    
    per_var_errors.append({
        'Variable': var_name,
        'MSE': mse_i,
        'RMSE': rmse_i,
        'MAE': mae_i,
        'R²': 1 - (np.sum((X_np[:, i] - X_pred_np[:, i]) ** 2) / 
                  np.sum((X_np[:, i] - np.mean(X_np[:, i])) ** 2))
    })

errors_df = pd.DataFrame(per_var_errors)
print("\nPer-Variable Prediction Errors:")
print(errors_df.to_string(index=False))

In [ ]:
# Analyze indirect effects and confounding structures
print("=" * 80)
print("STRUCTURAL ANALYSIS OF CAUSAL GRAPH")
print("=" * 80)

# Find parents and children of ETA_mins
eta_idx = variable_names.index('ETA_mins')
parents = [variable_names[j] for j in range(n_vars) 
           if j != eta_idx and abs(B_learned[eta_idx, j]) > 0.05]
children = [variable_names[i] for i in range(n_vars) 
            if i != eta_idx and abs(B_learned[i, eta_idx]) > 0.05]

print(f"\nDirect Parents of ETA_mins (confounders/causes):")
if parents:
    for p in parents:
        print(f"  • {p}")
else:
    print("  None detected")

print(f"\nDirect Children of ETA_mins (effects):")
if children:
    for c in children:
        print(f"  • {c}")
else:
    print("  None detected")

# Find colliders (v-structures): X → Z ← Y
print(f"\nColliders (confounding structures X→Z←Y):")
colliders = []
for z in range(n_vars):
    var_z = variable_names[z]
    incoming = [j for j in range(n_vars) if j != z and abs(B_learned[z, j]) > 0.05]
    if len(incoming) >= 2:
        source_vars = [variable_names[j] for j in incoming]
        colliders.append({
            'Collider': var_z,
            'Incoming from': ' and '.join(source_vars),
            'Structure': f"{' and '.join(source_vars)} → {var_z}"
        })

if colliders:
    for c in colliders:
        print(f"  • {c['Structure']}")
else:
    print("  None detected")

In [ ]:
# Analyze direct effects on target variable (ETA_mins)
target_idx = variable_names.index('ETA_mins')

print("=" * 80)
print("CAUSAL EFFECTS ON ETA_mins (Delivery Time)")
print("=" * 80)

direct_effects = []
for j, source_var in enumerate(variable_names):
    if j != target_idx:
        effect_weight = B_learned[target_idx, j]
        direct_effects.append({
            'Variable': source_var,
            'Effect Weight': effect_weight,
            'Absolute Effect': abs(effect_weight),
            'Direction': 'Increases ETA' if effect_weight > 0 else 'Decreases ETA'
        })

effects_df = pd.DataFrame(direct_effects).sort_values('Absolute Effect', ascending=False)

print("\nDirect Causal Effects (sorted by magnitude):")
print(effects_df.to_string(index=False))

# Identify strongest predictors
print("\n" + "=" * 80)
print("Strongest Causal Predictors of ETA_mins:")
print("=" * 80)

strong_effects = effects_df[effects_df['Absolute Effect'] > 0.05]
if len(strong_effects) > 0:
    for idx, row in strong_effects.iterrows():
        print(f"\n• {row['Variable']}")
        print(f"  Effect Weight: {row['Effect Weight']:.4f}")
        print(f"  Impact: {row['Direction']}")
else:
    print("No strong direct effects detected (threshold: 0.05)")

In [ ]:
import networkx as nx

# Create directed graph from adjacency matrix
G = nx.DiGraph()

# Add all variables as nodes
for var_name in variable_names:
    G.add_node(var_name)

# Add edges with weights from learned adjacency matrix
for i, target in enumerate(variable_names):
    for j, source in enumerate(variable_names):
        if i != j and abs(B_learned[i, j]) > 0.05:  # Significant edge threshold
            G.add_edge(source, target, weight=B_learned[i, j])

# Visualize the causal graph
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Plot 1: Adjacency matrix heatmap
ax = axes[0]
sns.heatmap(B_df, annot=True, fmt='.3f', cmap='RdBu_r', center=0, 
            cbar_kws={'label': 'Edge Weight'}, ax=ax, vmin=-0.5, vmax=0.5)
ax.set_title('Learned Causal Adjacency Matrix (B)', fontsize=14, fontweight='bold')
ax.set_xlabel('Source Variable')
ax.set_ylabel('Target Variable')

# Plot 2: Network graph
ax = axes[1]

# Use hierarchical layout for DAG
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Separate ETA_mins (target) from other variables
target_var = 'ETA_mins'
other_vars = [v for v in variable_names if v != target_var]

# Adjustpos for better visualization - put target on the right
if target_var in pos:
    pos[target_var] = np.array([1.5, 0.5])
    for i, var in enumerate(other_vars):
        angle = 2 * np.pi * i / len(other_vars)
        pos[var] = np.array([np.cos(angle), np.sin(angle)])

# Draw nodes
node_colors = ['#ff7f0e' if node == target_var else '#1f77b4' 
               for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000, ax=ax)

# Draw edges with varying width based on weight
edges_list = G.edges()
if edges_list:
    weights = [abs(G[u][v]['weight']) for u, v in edges_list]
    max_weight = max(weights)
    edge_widths = [3 * w / max_weight for w in weights]
else:
    edge_widths = []

nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='gray', 
                       arrowsize=20, arrowstyle='->', ax=ax, 
                       connectionstyle='arc3,rad=0.1')

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)

ax.set_title('Learned Causal Graph\n(Orange: Target, Blue: Predictors)', 
             fontsize=14, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('dcdi_causal_graph.png', dpi=300, bbox_inches='tight')
plt.show()

print("Causal graph visualization completed")

In [ ]:
# Extract the learned causal graph
B_learned = model.get_B().detach().cpu().numpy()

print("Learned Adjacency Matrix (B):")
print("=" * 80)
B_df = pd.DataFrame(
    B_learned,
    index=variable_names,
    columns=variable_names
)
print(B_df.round(4))

print("\n" + "=" * 80)
print("Causal Edges (edges with significant weight > 0.1):")
print("=" * 80)

edges = []
for i, target in enumerate(variable_names):
    for j, source in enumerate(variable_names):
        if i != j and abs(B_learned[i, j]) > 0.05:  # Significant edge threshold
            edges.append({
                'source': source,
                'target': target,
                'weight': B_learned[i, j],
                'direction': f"{source} → {target}"
            })

if edges:
    edges_df = pd.DataFrame(edges).sort_values('weight', ascending=False)
    print(edges_df.to_string(index=False))
else:
    print("No significant causal edges detected")

In [ ]:
# Plot training losses
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(loss_history['total'], linewidth=2)
axes[0, 0].set_title('Total Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(loss_history['mse'], label='MSE', linewidth=2)
axes[0, 1].set_title('Reconstruction Loss (MSE)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MSE Loss')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(loss_history['l1'], label='L1', linewidth=2, color='orange')
axes[1, 0].set_title('Sparsity Loss (L1)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('L1 Loss')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(loss_history['dag'], label='DAG', linewidth=2, color='red')
axes[1, 1].set_title('DAG Constraint Loss', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('DAG Loss')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dcdi_training_losses.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training losses plotted")

In [ ]:
def dcdi_loss(X, X_pred, model, lambda_reg=0.1, lambda_dag=100):
    """DCDI loss function combining:
    1. Reconstruction loss (MSE)
    2. L1 regularization (sparsity)
    3. DAG constraint
    """
    # Reconstruction loss
    mse_loss = nn.MSELoss()(X, X_pred)
    
    # L1 regularization on adjacency matrix (promotes sparsity)
    B = model.get_B()
    l1_loss = lambda_reg * torch.norm(B, p=1)
    
    # DAG constraint
    dag_loss = lambda_dag * model.acyclic_constraint() ** 2
    
    total_loss = mse_loss + l1_loss + dag_loss
    
    return total_loss, mse_loss, l1_loss, dag_loss


def train_dcdi(model, X_tensor, epochs=100, lr=0.01, batch_size=64, 
               lambda_reg=0.1, lambda_dag=100):
    """Train DCDI model.
    
    Args:
        model: DCDI_Model instance
        X_tensor: Input data tensor
        epochs: Number of training epochs
        lr: Learning rate
        batch_size: Batch size for training
        lambda_reg: L1 regularization coefficient
        lambda_dag: DAG constraint coefficient
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Create data loader
    dataset = TensorDataset(X_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    loss_history = {'total': [], 'mse': [], 'l1': [], 'dag': []}
    
    for epoch in range(epochs):
        epoch_losses = {'total': 0, 'mse': 0, 'l1': 0, 'dag': 0}
        n_batches = 0
        
        for batch_data in dataloader:
            X_batch = batch_data[0]
            
            # Forward pass
            X_pred = model(X_batch)
            
            # Compute loss
            total_loss, mse_loss, l1_loss, dag_loss = dcdi_loss(
                X_batch, X_pred, model, lambda_reg=lambda_reg, lambda_dag=lambda_dag
            )
            
            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            # Accumulate losses
            epoch_losses['total'] += total_loss.item()
            epoch_losses['mse'] += mse_loss.item()
            epoch_losses['l1'] += l1_loss.item()
            epoch_losses['dag'] += dag_loss.item()
            n_batches += 1
        
        # Average losses over batches
        for key in epoch_losses:
            epoch_losses[key] /= n_batches
            loss_history[key].append(epoch_losses[key])
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | "
                  f"Total: {epoch_losses['total']:.4f} | "
                  f"MSE: {epoch_losses['mse']:.4f} | "
                  f"L1: {epoch_losses['l1']:.4f} | "
                  f"DAG: {epoch_losses['dag']:.4f}")
    
    return loss_history


# Train the model
print("Starting DCDI training...\n")
loss_history = train_dcdi(
    model, X_tensor, 
    epochs=150, 
    lr=0.01, 
    batch_size=32,
    lambda_reg=0.05,
    lambda_dag=100
)

print("\nTraining completed!")

In [ ]:
class DCDI_Model(nn.Module):
    """Differentiable Causal Discovery with Instrumental Variables
    
    Learns both the causal structure (DAG) and the causal mechanisms.
    """
    
    def __init__(self, n_vars, hidden_dim=32):
        super(DCDI_Model, self).__init__()
        self.n_vars = n_vars
        self.hidden_dim = hidden_dim
        
        # Adjacency matrix (DAG): B[i,j] represents edge from j to i
        # Initialized as identity (no causal relationships)
        self.B_raw = nn.Parameter(torch.eye(n_vars) * 0.1)
        
        # Causal mechanisms: one MLP per variable
        # Each MLP takes inputs from its causal parents and predicts the variable value
        self.mechanisms = nn.ModuleList()
        for i in range(n_vars):
            mechanism = nn.Sequential(
                nn.Linear(n_vars, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1)
            )
            self.mechanisms.append(mechanism)
            
    def get_B(self):
        """Get the adjacency matrix with DAG constraint.
        
        Uses element-wise thresholding to enforce sparsity.
        """
        # Soft threshold: elements with small magnitude are set to 0
        B = torch.where(
            torch.abs(self.B_raw) > 0.1,
            self.B_raw,
            torch.zeros_like(self.B_raw)
        )
        
        # Zero out the diagonal (no self-loops)
        B = B * (1 - torch.eye(self.n_vars, device=B.device))
        
        return B
    
    def acyclic_constraint(self):
        """DAG constraint: trace(exp(B^T * B)) - n_vars should be close to 0.
        
        This ensures the graph is acyclic by penalizing cycles.
        """
        B = self.get_B()
        # Matrix power approximation using eigendecomposition
        E = torch.matrix_power(torch.eye(self.n_vars, device=B.device) + B * B / self.n_vars, self.n_vars)
        return torch.trace(E) - self.n_vars
    
    def forward(self, X):
        """Forward pass: generate predictions using causal mechanisms.
        
        Args:
            X: Input data (batch_size, n_vars)
            
        Returns:
            X_pred: Predicted values (batch_size, n_vars)
        """
        B = self.get_B()
        batch_size = X.shape[0]
        
        X_pred = []
        for i in range(self.n_vars):
            # Get causal parents: variables that have edges pointing to variable i
            # B[i, :] tells us which variables are parents of variable i
            parent_mask = B[i, :] != 0
            
            if parent_mask.sum() > 0:
                # Use MLP to compute variable i's value based on its parents
                X_parents = X * parent_mask.float()
                pred_i = self.mechanisms[i](X_parents)
            else:
                # If no parents, use base mechanism initialized to near-zero
                pred_i = self.mechanisms[i](X) * 0  # Placeholder
                
            X_pred.append(pred_i)
        
        X_pred = torch.cat(X_pred, dim=1)
        return X_pred


# Initialize model
model = DCDI_Model(n_vars=n_vars, hidden_dim=32).to(device)

print("DCDI Model created")
print(f"Number of variables: {n_vars}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")
print(f"\nModel:\n{model}")

In [ ]:
# Prepare data for DCDI
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(X_scaled).to(device)

print(f"Data shape: {X_tensor.shape}")
print(f"Data type: {X_tensor.dtype}")
print(f"Device: {X_tensor.device}")

# Get variable names
variable_names = df.columns.tolist()
n_vars = len(variable_names)

print(f"\nVariables ({n_vars}):")
for i, name in enumerate(variable_names):
    print(f"  {i}: {name}")

In [ ]:
# Verify and prepare data for DCDI training
print("\n" + "=" * 80)
print("DATA PREPARATION FOR DCDI")
print("=" * 80)

print(f"\nDataset Information:")
print(f"  • Total samples: {len(df)}")
print(f"  • Number of variables: {len(df.columns)}")
print(f"  • Variables: {df.columns.tolist()}")

print(f"\nChecking data quality...")

# Check for any remaining missing values
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print(f"⚠ Warning: Found missing values")
    print(missing_values[missing_values > 0])
else:
    print(f"✓ No missing values detected")

# Check for infinite values
inf_mask = np.isinf(df.select_dtypes(include=[np.number]))
inf_count = inf_mask.sum().sum()
if inf_count > 0:
    print(f"⚠ Warning: Found {inf_count} infinite values")
else:
    print(f"✓ No infinite values detected")

# Display sample rows
print(f"\nFirst 10 rows of extracted data:")
print(df.head(10))

print(f"\nDataset is ready for DCDI analysis!")

In [ ]:
# Load the feature data from featured_engineered directory
# Reads features.csv that contains pre-engineered features

import os
from datetime import datetime

featured_engineered_path = "../dataset/ml/PROCESSED/featured_engineered"
features_csv_path = os.path.join(featured_engineered_path, "features.csv")

print("Loading pre-engineered features from CSV...")
print("=" * 80)

# Load features.csv
if os.path.exists(features_csv_path):
    features_df = pd.read_csv(features_csv_path)
    
    file_size = os.path.getsize(features_csv_path) / (1024**2)  # Convert to MB
    file_time = os.path.getmtime(features_csv_path)
    file_date = datetime.fromtimestamp(file_time).strftime("%Y-%m-%d %H:%M:%S")
    
    print(f"✓ Successfully loaded: {os.path.basename(features_csv_path)}")
    print(f"  Size: {file_size:.2f} MB | Modified: {file_date}")
    print(f"\nDataset shape: {features_df.shape}")
    print(f"\nAvailable columns ({len(features_df.columns)}):")
    for i, col in enumerate(features_df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\nData types:")
    print(features_df.dtypes)
    
    print(f"\nFirst few rows:")
    print(features_df.head())
    
    print(f"\nData summary:")
    print(features_df.describe())
else:
    raise FileNotFoundError(f"features.csv not found at: {features_csv_path}\n"
                           f"Please ensure features.csv exists in: {featured_engineered_path}")

In [ ]:
# Extract relevant features for DCDI analysis
print("\n" + "=" * 80)
print("EXTRACTING FEATURES FOR CAUSAL ANALYSIS")
print("=" * 80)

# Define the features we need for DCDI
causal_features = [
    'batch_rank_capped',
    'speed_mean',
    'gps_gap_min',
    'spatial_congestion_norm',
    'WSI',  # Weather Severity Index
    'typecode_grouped',
    'eta_mins'  # Target variable
]

print("\nLooking for causal analysis features...")

# Check which features are available (case-insensitive)
available_cols = features_df.columns.tolist()
matching_features = []
feature_mapping = {}

for feature in causal_features:
    feature_lower = feature.lower()
    matches = [col for col in available_cols if feature_lower in col.lower()]
    
    if matches:
        matched_col = matches[0]
        matching_features.append(matched_col)
        feature_mapping[matched_col] = feature
        print(f"  ✓ {feature:<25} → {matched_col}")
    else:
        print(f"  ✗ {feature:<25} (not found)")

# Select and prepare the features for DCDI
if matching_features:
    df = features_df[matching_features].copy()
    
    # Rename columns to standard names
    df = df.rename(columns={v: k for k, v in feature_mapping.items()})
    
    # Drop rows with missing values
    initial_rows = len(df)
    df = df.dropna()
    dropped_rows = initial_rows - len(df)
    
    if dropped_rows > 0:
        print(f"\n⚠ Dropped {dropped_rows} rows with missing values")
    
    print(f"\n✓ Extracted {len(matching_features)} features")
    print(f"Data shape: {df.shape}")
    
    print(f"\nFeature statistics:")
    print(df.describe())
    
    print(f"\nFeature data types:")
    print(df.dtypes)
else:
    raise ValueError("No matching features found for DCDI analysis. "
                    "Please check column names in features.csv")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")